In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as patches
import src.get_data_4 as gd
from scipy.ndimage import gaussian_filter

In [ ]:
# ---------------------------
# Arquitectura: U-Net compacto (DIP)
# ---------------------------

def make_generator_m(n, m, in_channels, out_channels=1, base_filters=64):
    """
    Generador U-Net con skip connections.

    Parámetros
    ----------
    n, m        : dimensiones espaciales de la entrada (y de la salida)
    in_channels : número de canales de entrada = C_noise + 1 (ruido + B_n)
    out_channels: canales de salida (1 por defecto)
    base_filters: filtros en el primer bloque del encoder
    """
    inp = layers.Input(shape=(n, m, in_channels))

    # --- Encoder ---
    e1 = layers.Conv2D(base_filters, 3, padding="same")(inp)
    e1 = layers.LeakyReLU(0.2)(e1)
    e1 = layers.Conv2D(base_filters, 3, padding="same")(e1)
    e1 = layers.LeakyReLU(0.2)(e1)
    p1 = layers.AveragePooling2D(pool_size=(2, 2))(e1)

    e2 = layers.Conv2D(base_filters * 2, 3, padding="same")(p1)
    e2 = layers.LeakyReLU(0.2)(e2)
    e2 = layers.Conv2D(base_filters * 2, 3, padding="same")(e2)
    e2 = layers.LeakyReLU(0.2)(e2)
    p2 = layers.AveragePooling2D(pool_size=(2, 2))(e2)

    # --- Bottleneck ---
    b = layers.Conv2D(base_filters * 4, 3, padding="same")(p2)
    b = layers.LeakyReLU(0.2)(b)
    b = layers.Conv2D(base_filters * 4, 3, padding="same")(b)
    b = layers.LeakyReLU(0.2)(b)

    # --- Decoder con skip connections ---
    u1 = layers.UpSampling2D()(b)
    u1 = layers.Concatenate()([u1, e2])
    d1 = layers.Conv2D(base_filters * 2, 3, padding="same")(u1)
    d1 = layers.LeakyReLU(0.2)(d1)
    d1 = layers.Conv2D(base_filters * 2, 3, padding="same")(d1)
    d1 = layers.LeakyReLU(0.2)(d1)

    u2 = layers.UpSampling2D()(d1)
    u2 = layers.Concatenate()([u2, e1])
    d2 = layers.Conv2D(base_filters, 3, padding="same")(u2)
    d2 = layers.LeakyReLU(0.2)(d2)
    d2 = layers.Conv2D(base_filters, 3, padding="same")(d2)
    d2 = layers.LeakyReLU(0.2)(d2)

    out = layers.Conv2D(out_channels, 1, padding="same", activation="linear")(d2)

    return Model(inputs=inp, outputs=out)


# ---------------------------
# Pérdida de variación total
# ---------------------------

def total_variation_loss(x):
    """Variación total isotrópica. x: tensor (batch, H, W, 1)."""
    dh = tf.abs(x[:, 1:, :, :] - x[:, :-1, :, :])
    dw = tf.abs(x[:, :, 1:, :] - x[:, :, :-1, :])
    return tf.reduce_mean(dh) + tf.reduce_mean(dw)


In [ ]:
def run_dip_partial_boundary(cheap, boundary, mask,
                             noise_channels=16,
                             iters=5000,
                             lr=1e-3,
                             weight_cheap=0.1,
                             weight_tv=1e-4,
                             print_every=200):
    """
    Reconstrucción DIP con B_n como canal de entrada.

    Parámetros
    ----------
    cheap          : matriz barata completa (n, m)
    boundary       : valores conocidos de MC, resto 0  (n, m)
    mask           : 1 en puntos conocidos, 0 en el resto (n, m)
    noise_channels : canales de ruido aleatorio (C_noise)
    iters          : iteraciones máximas
    lr             : learning rate de Adam
    weight_cheap   : peso de L_cheap en zonas no observadas
    weight_tv      : peso de la pérdida de variación total
    print_every    : frecuencia de impresión
    """
    n, m = cheap.shape
    cheap    = cheap.astype(np.float32)
    boundary = boundary.astype(np.float32)
    mask     = mask.astype(np.float32)

    # --- Normalización ---
    s = max(np.max(np.abs(cheap)), np.max(np.abs(boundary[mask == 1])) if mask.any() else 0.0)
    if s == 0.0:
        s = 1.0
    cheap_n    = cheap    / s
    boundary_n = boundary / s
    # mask no se escala

    # --- Ruido fijo z: (1, n, m, C_noise) ---
    z = np.random.normal(size=(1, n, m, noise_channels)).astype(np.float32)

    # --- B_n como canal extra: (1, n, m, 1) ---
    B_channel = cheap_n[np.newaxis, :, :, np.newaxis]   # (1, n, m, 1)

    # --- Generador: entrada = C_noise + 1 canales ---
    in_channels = noise_channels + 1
    gen = make_generator_m(n, m, in_channels=in_channels, base_filters=64)
    optimizer = optimizers.Adam(learning_rate=lr)

    # --- Historial y early stopping ---
    history = {"loss_total": [], "loss_mc": [], "loss_cheap": [], "loss_tv": []}
    best_loss = np.inf
    best_pred = None
    patience  = max(200, int(0.15 * iters))  # 15% de iters, mínimo 200
    wait      = 0
    ema_alpha = 0.05   # suavizado: ventana efectiva ~1/alpha iteraciones
    ema_loss  = None   # se inicializa en la primera iteración

    mask_tf      = tf.constant(mask[np.newaxis, :, :, np.newaxis],      dtype=tf.float32)
    mask_inv_tf  = tf.constant((1.0 - mask)[np.newaxis, :, :, np.newaxis], dtype=tf.float32)
    boundary_tf  = tf.constant(boundary_n[np.newaxis, :, :, np.newaxis], dtype=tf.float32)
    cheap_tf     = tf.constant(cheap_n[np.newaxis, :, :, np.newaxis],    dtype=tf.float32)
    n_omega      = float(mask.sum())
    n_omega_c    = float((1.0 - mask).sum())

    for it in range(1, iters + 1):
        # Jitter solo sobre el ruido, B_n es determinístico
        z_jitter  = z + 0.03 * np.random.normal(size=z.shape).astype(np.float32)
        inp_batch = np.concatenate([z_jitter, B_channel], axis=-1)   # (1, n, m, C_noise+1)

        with tf.GradientTape() as tape:
            out = gen(inp_batch, training=True)   # (1, n, m, 1)

            # L_mc : MSE en puntos conocidos (Omega)
            loss_mc    = tf.reduce_sum(tf.square((out - boundary_tf) * mask_tf)) / (n_omega + 1e-8)

            # L_cheap : MSE en zonas no observadas (Omega^c)
            loss_cheap = tf.reduce_sum(tf.square((out - cheap_tf) * mask_inv_tf)) / (n_omega_c + 1e-8)

            # L_tv
            loss_tv = total_variation_loss(out)

            loss = loss_mc + weight_cheap * loss_cheap + weight_tv * loss_tv

        grads = tape.gradient(loss, gen.trainable_variables)
        optimizer.apply_gradients(zip(grads, gen.trainable_variables))

        lmc_val  = float(loss_mc.numpy())
        history["loss_total"].append(float(loss.numpy()))
        history["loss_mc"].append(lmc_val)
        history["loss_cheap"].append(float(loss_cheap.numpy()))
        history["loss_tv"].append(float(loss_tv.numpy()))

        # EMA de loss_mc para suavizar fluctuaciones del jitter
        if ema_loss is None:
            ema_loss = lmc_val
        else:
            ema_loss = ema_alpha * lmc_val + (1.0 - ema_alpha) * ema_loss

        # Early stopping: guardamos el mejor modelo según la EMA
        if ema_loss < best_loss:
            best_loss = ema_loss
            best_pred = out.numpy()[0, :, :, 0].copy()
            wait = 0
        else:
            wait += 1

        if it % print_every == 0 or it == 1:
            print(f"it {it:05d} | loss={history['loss_total'][-1]:.4e} "                  f"| mc={lmc_val:.4e}  cheap={history['loss_cheap'][-1]:.4e}  tv={history['loss_tv'][-1]:.4e}")

        if wait > patience:
            print(f"Early stopping en iteración {it} (sin mejora en loss_mc).")
            break

    # --- Reconstrucción final ---
    out_final = best_pred if best_pred is not None else out.numpy()[0, :, :, 0]
    out_final = out_final * s

    # Post-procesado: forzar valores exactos en puntos conocidos
    out_final[mask == 1] = boundary[mask == 1]

    return out_final, history


In [ ]:
df = pd.read_csv("SOC_matrix_g-2.csv", header=None)  # usa la primera columna como índice
SOC_matrix = df.values

In [ ]:
path = './datos_sqr-2/'
coord, soc, params = gd.get_points(path)

In [ ]:
n, m = SOC_matrix.shape

known_points = coord
known_values = soc

# Construir máscara y boundary
mask     = np.zeros((n, m), dtype=np.float32)
boundary = np.zeros((n, m), dtype=np.float32)

for (i, j), val in zip(known_points, known_values):
    mask[i, j]     = 1.0
    boundary[i, j] = val

# Matriz barata (puede suavizarse)
cheap          = SOC_matrix.copy().astype(np.float32)
cheap_smoothed = gaussian_filter(cheap, sigma=1.0)

# Ejecutar DIP
recon, hist = run_dip_partial_boundary(
    cheap_smoothed, boundary, mask,
    noise_channels=16,
    iters=5000,
    lr=1e-3,
    weight_cheap=0.1,
    weight_tv=1e-4,
)


In [ ]:
# ---------------------------
# Diagnóstico: evolución de pérdidas
# ---------------------------

def plot_loss_history(history, iters_run=None):
    """
    Grafica la evolución de cada componente de la pérdida.
    Muestra también la EMA de loss_mc para visualizar el criterio
    de early stopping.
    """
    its = np.arange(1, len(history["loss_total"]) + 1)

    # Reconstruir EMA con el mismo alpha usado en entrenamiento
    ema_alpha = 0.05
    ema = []
    val = None
    for v in history["loss_mc"]:
        val = v if val is None else ema_alpha * v + (1.0 - ema_alpha) * val
        ema.append(val)
    ema = np.array(ema)

    # Iteración donde se guardó el mejor modelo (mínimo de la EMA)
    best_it = int(np.argmin(ema)) + 1

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Panel izquierdo: todas las pérdidas
    ax = axes[0]
    ax.semilogy(its, history["loss_total"],  label="loss total",  alpha=0.7)
    ax.semilogy(its, history["loss_mc"],     label="loss MC (Ω)", alpha=0.7)
    ax.semilogy(its, history["loss_cheap"],  label="loss cheap",  alpha=0.7)
    ax.semilogy(its, history["loss_tv"],     label="loss TV",     alpha=0.7)
    ax.axvline(best_it, color="black", linestyle="--", linewidth=1.0, label=f"mejor it={best_it}")
    ax.set_xlabel("Iteración")
    ax.set_ylabel("Pérdida (escala log)")
    ax.set_title("Historial de pérdidas")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)

    # Panel derecho: loss_mc cruda vs EMA
    ax = axes[1]
    ax.semilogy(its, history["loss_mc"], alpha=0.4, label="loss MC cruda")
    ax.semilogy(its, ema,                 linewidth=2, label=f"EMA (α=0.05)")
    ax.axvline(best_it, color="black", linestyle="--", linewidth=1.0, label=f"mejor it={best_it}")
    ax.set_xlabel("Iteración")
    ax.set_ylabel("loss MC (escala log)")
    ax.set_title("Early stopping: loss MC vs EMA")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Iteraciones ejecutadas : {len(its)}")
    print(f"Mejor iteración (EMA)  : {best_it}")
    print(f"loss_mc en mejor it    : {history['loss_mc'][best_it-1]:.4e}")
    print(f"EMA en mejor it        : {ema[best_it-1]:.4e}")

plot_loss_history(hist)


In [ ]:
pts = np.array(params)

# noval = np.array([[-2.016, -0.268], [-1.354, 0.11], [-1.354, -1.402], [-1.685, -0.835]])

In [ ]:
plt.figure(figsize=(12,4))
extent = [-4, 2, -4, 2]
x0, y0 = -2.5, -1.5    
lado = 3

# Primer panel
ax1 = plt.subplot(1,3,1)
ax1.set_title('Numeric')
im1 = ax1.imshow(cheap.T, origin='lower', extent=extent)
ax1.scatter(pts[:,0], pts[:,1], color='red', s=20)
# ax1.scatter(noval[:,0], noval[:,1], color='green', s=20)
rect = patches.Rectangle(
    (x0, y0), lado, lado,
    linewidth=1, edgecolor='red', facecolor='none', angle=0
)
ax1.add_patch(rect)
plt.colorbar(im1, ax=ax1)

# Segundo panel
ax2 = plt.subplot(1,3,2)
ax2.set_title('IA[KMC] (DIP)')
im2 = ax2.imshow(recon.T, origin='lower', extent=extent)
plt.colorbar(im2, ax=ax2)

# Tercer panel
ax3 = plt.subplot(1,3,3)
ax3.set_title('diff (IA - Numeric)')
im3 = ax3.imshow(np.abs(recon - cheap).T, origin='lower', extent=extent)
ax3.scatter(pts[:,0], pts[:,1], color='red', s=10)
# ax1.scatter(noval[:,0], noval[:,1], color='green', s=20)
rect = patches.Rectangle(
    (x0, y0), lado, lado,
    linewidth=1, edgecolor='red', facecolor='none', angle=0
)
ax3.add_patch(rect)
plt.colorbar(im3, ax=ax3)

plt.tight_layout()
plt.show()